# Gold — Fato de Uber

Esta tabela representa a camada analítica detalhada das corridas corporativas de Uber.

A fato especializada preserva atributos específicos como origem, destino e data da corrida, que não pertencem à fato consolidada de viagens.

In [0]:
from pyspark.sql import functions as F

df_uber_silver = spark.table(
    "viagens_teste.silver.uber"
)

print(
    "Registros Silver:",
    df_uber_silver.count()
)

### Construção da fato de Uber

Selecionamos os atributos necessários para análises financeiras, temporais e operacionais das corridas corporativas.

In [0]:
df_fato_uber = (
    df_uber_silver
    .select(
        F.col("id_corrida"),
        F.col("viajante"),
        F.col("data_corrida"),

        F.col("centro_custo"),
        F.col("diretoria"),

        F.col("origem"),
        F.col("destino"),

        F.col("valor_numerico").alias("valor"),

        F.col("agencia"),
        F.col("trimestre"),

        F.lit("Uber").alias("tipo_despesa")
    )
)

In [0]:
display(df_fato_uber)

### Validação da Gold

Verificamos a quantidade de corridas e a presença dos principais atributos antes da persistência.

In [0]:
print(
    "Total de corridas:",
    df_fato_uber.count()
)

print(
    "Origem ausente:",
    df_fato_uber
    .filter(F.col("origem").isNull())
    .count()
)

print(
    "Destino ausente:",
    df_fato_uber
    .filter(F.col("destino").isNull())
    .count()
)

print(
    "Centro de custo ausente:",
    df_fato_uber
    .filter(F.col("centro_custo").isNull())
    .count()
)

In [0]:
(
    df_fato_uber.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "viagens_teste.gold.fato_uber"
    )
)

In [0]:
display(
    spark.table(
        "viagens_teste.gold.fato_uber"
    )
)